1) Output the first 5 rows of data of your own variant
2) Split the data so that the test part contains 29% of the data, and the remaining part is in the training part.
3) To train the classifier, use RandomForestClassifier from the pyspark.ml.classification package and MulticlassClassificationEvaluator from the pyspark.ml.evaluation package. Build a classifier that determines the product category based on f1,…f5. In RandomForestClassifier, specify the number of trees using the numTrees parameter (14), and in MulticlassClassificationEvaluator, specify metricName="accuracy".
4) Analyze the importance of each feature. Find a feature whose removal results in a drop in quality of no more than 10%.

In [16]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [17]:
spark = SparkSession.builder.appName("ClassificationExample").getOrCreate()
df = spark.read.csv("30.csv", header=True, inferSchema=True)
df.show(5)

+---+-----+-----+-----+-----+-----+--------+
| id|   f1|   f2|   f3|   f4|   f5|category|
+---+-----+-----+-----+-----+-----+--------+
|  0|-1.49| 4.46|-4.57| -2.8|-2.49|       1|
|  1| 2.48| 1.48|-0.68|-1.34| 4.29|       4|
|  2|-3.65| 1.43| 2.46|-0.21| 1.54|       2|
|  3|-2.62| 4.35|-0.17|-0.73|-4.04|       1|
|  4|-3.97|-2.09| 2.73|  2.0| 0.11|       2|
+---+-----+-----+-----+-----+-----+--------+
only showing top 5 rows


In [18]:
assembler = VectorAssembler(inputCols=['f1', 'f2', 'f3', 'f4', 'f5'], outputCol='features')
assembled_df = assembler.transform(df)

In [19]:
train_df, test_df = assembled_df.randomSplit([0.71, 0.29], seed=42)

In [20]:
rfc = RandomForestClassifier(labelCol='category', featuresCol='features', numTrees=14, seed=42)
model = rfc.fit(train_df)

In [22]:
predictions = model.transform(test_df)
predictions.show(5)

+---+-----+----+-----+-----+-----+--------+--------------------+--------------------+--------------------+----------+
| id|   f1|  f2|   f3|   f4|   f5|category|            features|       rawPrediction|         probability|prediction|
+---+-----+----+-----+-----+-----+--------+--------------------+--------------------+--------------------+----------+
|  2|-3.65|1.43| 2.46|-0.21| 1.54|       2|[-3.65,1.43,2.46,...|[0.0,0.0456597222...|[0.0,0.0032614087...|       3.0|
|  6| 1.26|4.65|-4.97| 2.95|  3.1|       3|[1.26,4.65,-4.97,...|[0.0,0.0291262135...|[0.0,0.0020804438...|       3.0|
|  8| 2.73|-1.9|-1.27| 1.37| 2.94|       3|[2.73,-1.9,-1.27,...|[0.0,0.0291262135...|[0.0,0.0020804438...|       3.0|
|  9| 0.66|2.08| 2.11| 2.13|-3.87|       1|[0.66,2.08,2.11,2...|[10.7289051092258...|[0.76635036494470...|       0.0|
| 13| 4.28|-4.9|-1.12|-3.25|-2.48|       1|[4.28,-4.9,-1.12,...|[1.23949765203345...|[0.08853554657381...|       1.0|
+---+-----+----+-----+-----+-----+--------+-------------

In [37]:
eval = MulticlassClassificationEvaluator(labelCol="category", predictionCol="prediction", metricName="accuracy")
accuracy = eval.evaluate(predictions)
print(f"{accuracy:.2f}")

0.86


In [26]:
model.featureImportances

SparseVector(5, {0: 0.0378, 1: 0.0221, 2: 0.0147, 3: 0.0319, 4: 0.8935})

In [28]:
assembler_new = VectorAssembler(inputCols=['f1', 'f2', 'f4', 'f5'], outputCol='features_reduced')
assembled_df_new = assembler_new.transform(df)

In [29]:
train_df_new, test_df_new = assembled_df_new.randomSplit([0.71, 0.29], seed=42)

In [31]:
rfc_new = RandomForestClassifier(labelCol='category', featuresCol='features_reduced', numTrees=14, seed=42)
model_new = rfc_new.fit(train_df_new)

In [33]:
predictions_new = model_new.transform(test_df_new)
accuracy_new = eval.evaluate(predictions_new)

In [36]:
print(f"Accuracy with all features: {accuracy:.4f}")
print(f"Accuracy without f3: {accuracy_new:.4f}")

Accuracy with all features: 0.8612
Accuracy without f3: 0.8531


In [39]:
drop = accuracy - accuracy_new
drop_percentage = (drop / accuracy) * 100 if accuracy > 0 else 0

print(f"{drop:.4f}")
print(f"{drop_percentage:.2f}%")

0.0082
0.95%
